# S6_6 — Ablation Studies

**Leaf-JEPA IRP** | Stage 6 — Analysis and Interpretation


**Purpose:** Decompose the system into its constituent contributions:
1. **A1**: Domain pretraining contribution (generic vs Leaf-JEPA, same PEFT)
2. **A2**: PEFT adaptation spectrum (LP → PEFT → Full FT)
3. **A3**: Masking strategy contribution (biased vs standard masking)

Each ablation answers: "what happens if we remove this component?"

**Outputs:**
- `ablation1_domain_pretraining.json` — generic vs Leaf-JEPA with PEFT
- `ablation2_adaptation_spectrum.png` — LP → PEFT → Full FT comparison
- `ablation3_masking_strategy.json` — biased vs standard masking
- `ablation3_convergence_comparison.png` — LP F1 during pretraining

**Research Questions Served:** RQ1 (A1, A3), RQ2/RQ3 (A2)

**Note:** A1 and A3 may require additional model training runs if not
already completed in Stages 4–5. A2 uses existing results only.


## Initialization

In [1]:
import sys, json
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from stage6_analysis_and_interpretation.config_stage6 import *
from stage6_analysis_and_interpretation.analysis_utils import (
    set_seed, load_json, save_json, print_section,
    cohens_d_paired, effect_size_label,
    paired_ttest, paired_wilcoxon,
)
import matplotlib.pyplot as plt

set_seed(RANDOM_SEED)
STAGE6_FIGURES.mkdir(parents=True, exist_ok=True)
STAGE6_DATA.mkdir(parents=True, exist_ok=True)


##  Ablation 1 — Domain Pretraining Contribution

In [ ]:
print_section("Ablation 1: Domain Pretraining Contribution")
print("Compares the SAME PEFT method applied to generic I-JEPA vs Leaf-JEPA.\n")

# Load results — adjust paths based on your Stage 5 output format
# You need: generic I-JEPA + PEFT results and Leaf-JEPA + PEFT results
# If you ran PEFT on both encoders in Stage 5, load those results here.
# If not, this ablation requires additional training runs.

a1_results = {"ablation": "domain_pretraining", "comparisons": []}

# Check if we have generic I-JEPA + PEFT results
generic_peft_path = PEFT_RESULTS_DIR / "generic_ijepa_peft_results.json"
leaf_peft_path = PEFT_RESULTS_DIR / "S1_method_comparison_summary.json"

if generic_peft_path.exists() and leaf_peft_path.exists():
    generic_results = load_json(generic_peft_path)
    leaf_results = load_json(leaf_peft_path)

    # Compare each PEFT method
    for method in PEFT_METHODS:
        g_data = generic_results.get(method, {})
        l_data = leaf_results.get(method, {})

        g_f1 = g_data.get("macro_f1_mean", None)
        l_f1 = l_data.get("macro_f1_mean", None)

        if g_f1 is not None and l_f1 is not None:
            delta = l_f1 - g_f1
            a1_results["comparisons"].append({
                "method": method,
                "generic_f1": g_f1,
                "leaf_f1": l_f1,
                "delta": delta,
                "interpretation": "Domain PT helped" if delta > 0 else "Domain PT did not help"
            })
            print(f"  {method}: Generic={g_f1:.4f}, Leaf={l_f1:.4f}, Delta={delta:+.4f}")
else:
    print("  \u26a0\ufe0f Generic I-JEPA + PEFT results not found.")
    print("  To complete Ablation 1, you need to:")
    print("  1. Load the generic I-JEPA checkpoint")
    print("  2. Apply the best PEFT method(s) with identical HPs")
    print("  3. Save results to generic_ijepa_peft_results.json")
    print("\n  Alternatively, use B3 vs B5 as a proxy (linear probe, no PEFT):")

    # Fallback: use B3 vs B5 comparison
    b3_path = BASELINES_DIR / "B3_aggregate.json"
    b5_path = BASELINES_DIR / "B5_aggregate.json"
    if b3_path.exists() and b5_path.exists():
        b3 = load_json(b3_path)
        b5 = load_json(b5_path)
        b3_f1 = b3.get("macro_f1_mean", b3.get("macro_f1", 0))
        b5_f1 = b5.get("macro_f1_mean", b5.get("macro_f1", 0))
        delta = b5_f1 - b3_f1
        a1_results["comparisons"].append({
            "method": "Linear Probe (proxy)",
            "generic_f1": b3_f1, "leaf_f1": b5_f1, "delta": delta,
        })
        print(f"  Linear Probe: B3={b3_f1:.4f}, B5={b5_f1:.4f}, Delta={delta:+.4f}")

save_json(a1_results, STAGE6_DATA / "ablation1_domain_pretraining.json")


## Ablation 2 — PEFT Adaptation Spectrum

In [ ]:
print_section("Ablation 2: Adaptation Spectrum (LP -> PEFT -> Full FT)")
print("Shows the spectrum from no adaptation to full adaptation on Leaf-JEPA.\n")

spectrum = {}

# Linear probe (B5)
b5_path = BASELINES_DIR / "B5_aggregate.json"
if b5_path.exists():
    b5 = load_json(b5_path)
    spectrum["Linear Probe"] = {
        "f1": b5.get("macro_f1_mean", b5.get("macro_f1", 0)),
        "params": EMBED_DIM * NUM_CLASSES,  # ~48K
    }

# PEFT methods
s1_path = PEFT_RESULTS_DIR / "S1_method_comparison_summary.json"
if s1_path.exists():
    s1 = load_json(s1_path)
    if isinstance(s1, dict):
        for method, data in s1.items():
            if isinstance(data, dict):
                spectrum[method] = {
                    "f1": data.get("macro_f1_mean", data.get("macro_f1", 0)),
                    "params": data.get("trainable_params", data.get("params", 0)),
                }

# Full fine-tune (B4 or Leaf-JEPA full FT)
b4_path = BASELINES_DIR / "B4_aggregate.json"
if b4_path.exists():
    b4 = load_json(b4_path)
    spectrum["Full Fine-Tune"] = {
        "f1": b4.get("macro_f1_mean", b4.get("macro_f1", 0)),
        "params": 632_000_000,
    }

if spectrum:
    # Sort by params
    sorted_methods = sorted(spectrum.items(), key=lambda x: x[1]["params"])

    print("  Adaptation Spectrum:")
    for name, data in sorted_methods:
        print(f"    {name:<25} params={data['params']:>12,}   F1={data['f1']:.4f}")

    # PEFT efficiency ratio
    lp_f1 = spectrum.get("Linear Probe", {}).get("f1", 0)
    ft_f1 = spectrum.get("Full Fine-Tune", {}).get("f1", 0)
    ft_gap = ft_f1 - lp_f1

    if ft_gap > 0:
        print(f"\n  Full FT gap (FT - LP): {ft_gap:.4f}")
        for name, data in sorted_methods:
            if name not in ("Linear Probe", "Full Fine-Tune"):
                ratio = (data["f1"] - lp_f1) / ft_gap
                pct_params = data["params"] / 632_000_000 * 100
                print(f"    {name}: captures {ratio:.1%} of FT gap with {pct_params:.2f}% params")

    # Plot
    fig, ax = plt.subplots(figsize=(10, 6))
    names = [n for n, _ in sorted_methods]
    f1s = [d["f1"] for _, d in sorted_methods]
    params = [d["params"] for _, d in sorted_methods]

    colours = [METHOD_COLOURS.get(n, "#888888") for n in names]
    bars = ax.bar(range(len(names)), f1s, color=colours, edgecolor="white", linewidth=0.5)

    # RQ3 threshold line
    if ft_f1:
        ax.axhline(ft_f1 - RQ3_THRESHOLD_PCT / 100, color="green", ls="--", alpha=0.5,
                   label=f"Within {RQ3_THRESHOLD_PCT}% of Full FT")

    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Macro-F1")
    ax.set_title("Adaptation Spectrum: Linear Probe -> PEFT -> Full Fine-Tune")
    ax.legend()

    # Annotate with param counts
    for i, (bar, p) in enumerate(zip(bars, params)):
        pct = p / 632_000_000 * 100
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f"{pct:.1f}%", ha="center", va="bottom", fontsize=7, color="#666")

    plt.tight_layout()
    fig.savefig(STAGE6_FIGURES / "ablation2_adaptation_spectrum.png")
    plt.close(fig)
    print("  Figure saved.")


## Ablation 3 — Masking Strategy Contribution

In [ ]:
print_section("Ablation 3: Masking Strategy (Biased vs Standard)")
print("Compares Leaf-JEPA trained with disease-biased masking vs standard masking.\n")
print("This is the validation of the PRIMARY NOVEL CONTRIBUTION.\n")

a3_results = {"ablation": "masking_strategy", "comparisons": []}

# Check if S4_5 standard masking checkpoint exists
if LEAF_JEPA_STANDARD_MASKING_CHECKPOINT.exists():
    print("  Standard masking checkpoint found!")

    # Load LP monitor histories from both pretraining runs
    biased_history_path = PRETRAIN_DIR / "lp_monitor_history.json"
    standard_history_path = PRETRAIN_DIR / "standard_masking" / "lp_monitor_history.json"

    if biased_history_path.exists() and standard_history_path.exists():
        biased_history = load_json(biased_history_path)
        standard_history = load_json(standard_history_path)

        biased_lp = [(h["pretrain_epoch"], h["lp_val_macro_f1"]) for h in biased_history]
        standard_lp = [(h["pretrain_epoch"], h["lp_val_macro_f1"]) for h in standard_history]

        best_biased = max(biased_lp, key=lambda x: x[1])
        best_standard = max(standard_lp, key=lambda x: x[1])

        delta = best_biased[1] - best_standard[1]
        a3_results["comparisons"].append({
            "metric": "LP val macro-F1",
            "biased_masking": best_biased[1],
            "biased_epoch": best_biased[0],
            "standard_masking": best_standard[1],
            "standard_epoch": best_standard[0],
            "delta": delta,
        })

        print(f"  Biased masking:   best LP F1 = {best_biased[1]:.4f} (epoch {best_biased[0]})")
        print(f"  Standard masking: best LP F1 = {best_standard[1]:.4f} (epoch {best_standard[0]})")
        print(f"  Delta: {delta:+.4f}")
        print(f"  Interpretation: {'Biased masking HELPS' if delta > 0 else 'Biased masking does NOT help'}")

        # Convergence comparison plot
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.plot([e for e, _ in biased_lp], [f for _, f in biased_lp],
                "o-", color="#27ae60", label="Disease-biased masking", markersize=4)
        ax.plot([e for e, _ in standard_lp], [f for _, f in standard_lp],
                "s-", color="#e74c3c", label="Standard masking", markersize=4)
        ax.set_xlabel("Pretraining Epoch")
        ax.set_ylabel("Linear Probe Val Macro-F1")
        ax.set_title("Masking Strategy Ablation: Convergence Comparison")
        ax.legend()
        plt.tight_layout()
        fig.savefig(STAGE6_FIGURES / "ablation3_convergence_comparison.png")
        plt.close(fig)
        print("  Convergence figure saved.")
    else:
        print("  \u26a0\ufe0f LP monitor histories not found for both variants")
        print(f"  Checked: {biased_history_path}")
        print(f"  Checked: {standard_history_path}")

    # If you also ran PEFT on both checkpoints, add that comparison here
    # biased_peft_path = PEFT_RESULTS_DIR / "biased_masking_peft.json"
    # standard_peft_path = PEFT_RESULTS_DIR / "standard_masking_peft.json"

else:
    print("  \u274c Standard masking checkpoint NOT found!")
    print(f"  Expected at: {LEAF_JEPA_STANDARD_MASKING_CHECKPOINT}")
    print("\n  This is the MOST IMPORTANT missing experiment.")
    print("  To complete it:")
    print("  1. In config_stage4.py, set ENABLE_BIASED_MASKING = False")
    print("  2. Run S4_3 pretraining with a new output directory")
    print("  3. Export the checkpoint via S4_6")
    print("  4. Set LEAF_JEPA_STANDARD_MASKING_CHECKPOINT in config_stage6.py")
    print("  5. Re-run this cell")

save_json(a3_results, STAGE6_DATA / "ablation3_masking_strategy.json")


## Ablation Summary table

In [ ]:
print_section("Ablation Summary")

summary_rows = []

# A1
a1 = load_json(STAGE6_DATA / "ablation1_domain_pretraining.json")
for comp in a1.get("comparisons", []):
    summary_rows.append({
        "Ablation": "A1: Domain Pretraining",
        "Comparison": f"{comp['method']}: Generic vs Leaf-JEPA",
        "Baseline": comp.get("generic_f1", "N/A"),
        "With Component": comp.get("leaf_f1", "N/A"),
        "Delta": comp.get("delta", "N/A"),
    })

# A2: from spectrum
if spectrum:
    lp = spectrum.get("Linear Probe", {}).get("f1")
    ft = spectrum.get("Full Fine-Tune", {}).get("f1")
    for name, data in spectrum.items():
        if name not in ("Linear Probe", "Full Fine-Tune"):
            summary_rows.append({
                "Ablation": "A2: Adaptation Spectrum",
                "Comparison": f"{name} vs Full FT",
                "Baseline": data["f1"],
                "With Component": ft,
                "Delta": ft - data["f1"] if ft else "N/A",
            })

# A3
a3 = load_json(STAGE6_DATA / "ablation3_masking_strategy.json")
for comp in a3.get("comparisons", []):
    summary_rows.append({
        "Ablation": "A3: Masking Strategy",
        "Comparison": f"{comp['metric']}: Standard vs Biased",
        "Baseline": comp.get("standard_masking", "N/A"),
        "With Component": comp.get("biased_masking", "N/A"),
        "Delta": comp.get("delta", "N/A"),
    })

if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(STAGE6_TABLES / "ablation_summary.csv", index=False)
    print(summary_df.to_string(index=False))
else:
    print("  No ablation results to summarise yet.")

print("\n\u2705 S6_6 complete.")


## Critical Analysis

### Ablation 3 is Non-Negotiable
Without the masking ablation, the examiner cannot distinguish:
- "Domain pretraining helped" (expected, incremental contribution)
- "Disease-biased masking specifically helped" (your novel contribution)

If A3 shows the biased masking adds even 1-2 percentage points over standard masking,
that is a legitimate research contribution. If it shows no improvement, that is ALSO
a legitimate finding — document it honestly.

### Interpreting Ablation 2
The PEFT efficiency ratio tells you how much of the full fine-tuning gap PEFT closes:
- ratio ≈ 1.0 → PEFT captures nearly all gains → RQ3 confirmed
- ratio ≈ 0.5 → PEFT captures half → PEFT is useful but limited
- ratio < 0.3 → PEFT barely helps → the encoder features aren't easily adaptable

### Future Work from Ablations
Each ablation that shows a limitation suggests specific future work:
- A1 shows small delta → domain pretraining dataset may be too small
- A2 shows large gap → more expressive PEFT methods needed
- A3 shows no masking benefit → masking strategy needs refinement
